# Day 2-02：入力を変えて分類器を騙す

モデルを固定し、画像の画素値を更新します。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. 学習済みモデルと画像

01で保存した重みを読み込みます。まず変更前に正しく分類できているか確認します。

In [ ]:
from jkj1a.classification import load_classifier
from jkj1a.classification import load_cifar10
import torch
from torch.nn import functional as F

# 01で学習したモデルを読み込む
model = load_classifier()
device = next(model.parameters()).device

# 攻撃する画像を取り出す
_, test_loader = load_cifar10(batch_size=32)
x, y = next(iter(test_loader))
x, y = x.to(device), y.to(device)

with torch.no_grad():
    correct = model(x).argmax(1) == y
print("変更前の正解数:", correct.sum().item(), "/", len(y))
if not correct.any():
    raise ValueError("このバッチでは全問不正解です。別のバッチを選ぶか、01で学習を続けてください。")

# 変更前に正解している画像を表示用に選ぶ
index = correct.nonzero(as_tuple=True)[0][0].item()

## 2. FGSM：画像の勾配を使う

学習では重みを変えて損失を小さくしました。ここでは画像$x$を変えて、正解ラベル$y$に対する損失を大きくします。

$$x_{\mathrm{adv}}=\operatorname{clip}_{[0,1]}\left(x+\epsilon\,\operatorname{sign}(\nabla_x L(f(x),y))\right)$$

`eps=8/255`は、0〜1で表した各画素を最大8/255だけ変える制約です。正規化はモデルの内部で行います。

In [ ]:
from jkj1a.classification import CIFAR_CLASSES
from jkj1a.adversarial import show_attack

eps = 8/255

# 画像に関する勾配を計算する
x_input = x.detach().clone().requires_grad_(True)
loss = F.cross_entropy(model(x_input), y)
grad, = torch.autograd.grad(loss, x_input)

# 損失が増える向きに画素を変更する
x_adv = (x_input + eps * grad.sign()).clamp(0, 1).detach()

show_attack(model, x, x_adv, CIFAR_CLASSES, index=index)

## 3. I-FGSM：小さな更新を繰り返す

`alpha`は1回の更新幅、`eps`は元画像から許される変化の上限です。毎回、元画像からの差を$[-\epsilon,\epsilon]$に戻します。

`targeted=False`では正解クラスの損失を増やします。`targeted=True`では指定した目標クラスの損失を減らします。モデルの重みは更新しません。

In [ ]:
def ifgsm(model, x, y, *, eps=8/255, alpha=2/255, n_steps=10, targeted=False):
    """L-infinity I-FGSM; x is in [0,1], y is the target for targeted attacks."""
    if eps < 0 or alpha < 0 or not isinstance(n_steps, int) or n_steps < 1:
        raise ValueError('eps and alpha must be nonnegative; n_steps must be a positive integer')
    was_training = model.training
    model.eval()

    # 許容範囲の基準となる元画像を残す
    original = x.detach()
    adv = original.clone()

    try:

        for _ in range(n_steps):

            # 現在の画像に対する勾配を求める
            adv.requires_grad_(True)
            loss = F.cross_entropy(model(adv), y)
            grad, = torch.autograd.grad(loss, adv)

            # 目標クラスへの攻撃では、損失を減らす向きに進む
            direction = -1 if targeted else 1
            adv = adv.detach() + direction * alpha * grad.sign()

            # 元画像からの変化と、画素値の範囲を制限する
            adv = original + (adv - original).clamp(-eps, eps)
            adv = adv.clamp(0, 1).detach()

    finally:

        # 呼び出し前のモードに戻す
        model.train(was_training)

    return adv

In [ ]:
x_adv = ifgsm(model, x, y, eps=8/255, alpha=2/255, n_steps=10)

show_attack(model, x, x_adv, CIFAR_CLASSES, index=index)
print("画素の最大変化:", (x_adv-x).abs().max().item())

## 4. 複数の画像で評価する

同じテスト画像を使って比較します。`adversarial_accuracy`は攻撃後の正解率です。`success_rate`は**変更前に正解していた画像のうち、攻撃後に不正解となった割合**です。初めから間違えていた画像は攻撃成功に数えません。

ここでは先頭8バッチを評価します。全体を調べる場合は`max_batches=None`にします。

In [ ]:
from jkj1a.adversarial import evaluate_attack

results = []

for eps in (0, 1/255, 2/255, 4/255, 8/255, 16/255):
    row = evaluate_attack(model, test_loader, attack=ifgsm,
                          eps=eps, alpha=eps/4, n_steps=10, max_batches=8)
    results.append({"eps": eps, **row})
    print(results[-1], flush=True)

## 演習

1. `results`から、攻撃強度と攻撃後の正解率のグラフを描いてください。
2. `eps=8/255`を固定し、反復回数1・5・10・20で結果を比較してください。`alpha`も記録してください。
3. 各画素に`[-eps, eps]`の一様乱数を加えた画像を作り、同じ画像群へのI-FGSMと比較してください。
4. `targeted=True`を使い、指定したクラスに分類させてください。元の正解と異なる目標クラスを選び、目標クラスになった割合を計算してください。

In [ ]:
# ここにコードを書く